## XGBoost Model Training with Tuned Parameters and Selected Features

In this notebook, I train several **XGBoost models** using different sets of hyperparameters obtained from various **hyperparameter tuning algorithms**. At the same time, I apply the **best-performing features** identified through **Recursive Feature Elimination (RFE)** to improve model performance and generalization.

#### The Best Performance is Optuna

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import json
import pickle

In [ ]:
train = pd.read_pickle('../data/train_set.pkl')

In [ ]:
train.Target = train.Target.astype(int)

In [ ]:
with open("../data/selected_features.json","r",) as file:
    features = json.load(file)

with open("../data/best_grid_search_params.json","r",) as file:
    hyperparameters_grid_search = json.load(file)

with open("../data/best_random_search_params.json","r",) as file:
    hyperparameters_random_search = json.load(file)

with open("../data/best_optuna_params.json","r",) as file:
    hyperparameters_optuna = json.load(file)

In [ ]:
X = train[features]
y = train["Target"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [ ]:
X_train.shape

In [ ]:
X_val.shape

### Model based on grid search hyperparameters

In [ ]:
model_grid_search = XGBClassifier(
    learning_rate=hyperparameters_grid_search["learning_rate"],
    n_estimators=hyperparameters_grid_search["n_estimators"],
    max_depth=hyperparameters_grid_search["max_depth"],
    gamma=hyperparameters_grid_search["gamma"],
    subsample=hyperparameters_grid_search["subsample"],
    colsample_bytree=hyperparameters_grid_search["colsample_bytree"],
    objective="binary:logistic",
    seed=27,
    n_jobs=-1,
)

model_grid_search.fit(X_train, y_train, eval_set=[(X_val, y_val)])

### Model based on random_search hyperparameters

In [ ]:
model_random_search = XGBClassifier(
    learning_rate=hyperparameters_random_search["learning_rate"],
    n_estimators=hyperparameters_random_search["n_estimators"],
    max_depth=hyperparameters_random_search["max_depth"],
    gamma=hyperparameters_random_search["gamma"],
    subsample=hyperparameters_random_search["subsample"],
    colsample_bytree=hyperparameters_random_search["colsample_bytree"],
    objective="binary:logistic",
    seed=27,
    n_jobs=-1,
)

model_random_search.fit(X_train, y_train, eval_set=[(X_val, y_val)])

### Model based on optuna hyperparameters

In [ ]:
model_optuna = XGBClassifier(
    learning_rate=hyperparameters_optuna["learning_rate"],
    n_estimators=hyperparameters_optuna["n_estimators"],
    max_depth=hyperparameters_optuna["max_depth"],
    gamma=hyperparameters_optuna["gamma"],
    subsample=hyperparameters_optuna["subsample"],
    colsample_bytree=hyperparameters_optuna["colsample_bytree"],
    objective="binary:logistic",
    seed=27,
    n_jobs=-1,
)

model_optuna.fit(X_train, y_train, eval_set=[(X_val, y_val)])

### check ROC_AUC scores 

##### model_grid_search

In [ ]:
pred = model_grid_search.predict_proba(X_train).T[1]
roc_auc_score(y_train, pred)

In [ ]:
pred = model_grid_search.predict_proba(X_val).T[1]
roc_auc_score(y_val, pred)

##### model_random_search

In [ ]:
pred = model_random_search.predict_proba(X_train).T[1]
roc_auc_score(y_train, pred)

In [ ]:
pred = model_random_search.predict_proba(X_val).T[1]
roc_auc_score(y_val, pred)

##### model_optuna

In [ ]:
pred = model_optuna.predict_proba(X_train).T[1]
roc_auc_score(y_train, pred)

In [ ]:
pred = model_optuna.predict_proba(X_val).T[1]
roc_auc_score(y_val, pred)

### Final Model training with final parameters and on full train data

In [ ]:
model = XGBClassifier(
    learning_rate=hyperparameters_optuna["learning_rate"],
    n_estimators=hyperparameters_optuna["n_estimators"],
    max_depth=hyperparameters_optuna["max_depth"],
    gamma=hyperparameters_optuna["gamma"],
    subsample=hyperparameters_optuna["subsample"],
    colsample_bytree=hyperparameters_optuna["colsample_bytree"],
    objective="binary:logistic",
    seed=27,
    n_jobs=-1,
)

model.fit(X, y)

In [ ]:
pred = model.predict_proba(X).T[1]
roc_auc_score(y, pred)

### Saved Final Model

In [ ]:
pickle.dump(model, open("../data/model.pkl", "wb"))

In [ ]:
import os
import bentoml

## Define path BentoML
os.environ["BENTOML_HOME"] = "../bentoml"

#Model save 
exp = bentoml.xgboost.save_model(
    f"credit_scoring:b9", model
)